In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import math

In [30]:
src_sentences = [
    ["hello", "world", "."],
    ["how", "are", "you", "?"],
    ["i", "am", "fine", "."],
    ["this", "is", "a", "test"],
    ["another", "example"]
]
trg_sentences = [
    ["hello", "world", "!"],
    ["how", "do", "you", "do", "?"],
    ["i", "am", "doing", "well", "."],
    ["this", "is", "a", "reference"],
    ["one", "more", "example"]
]

dataset = list(zip(src_sentences, trg_sentences))



In [31]:
def build_vocab(sentences, pad=0, sos=1, eos=2):
    words = set(word for sent in sentences for word in sent)
    vocab = {word: idx+3 for idx, word in enumerate(words)}
    vocab['<pad>'] = pad
    vocab['<sos>'] = sos
    vocab['<eos>'] = eos
    itos = {idx: word for word, idx in vocab.items()}
    return vocab, itos

SRC_vocab, SRC_itos = build_vocab(src_sentences)
TRG_vocab, TRG_itos = build_vocab(trg_sentences)

In [32]:
def tokens_to_words(tokens, itos):
    return [itos[t] for t in tokens if t in itos]



In [33]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim)
    def forward(self, src):
        embedded = self.embedding(src.unsqueeze(1))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim)
        self.fc_out = nn.Linear(hid_dim, output_dim)
    def forward(self, input, hidden, cell):
        embedded = self.embedding(input.unsqueeze(0))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    def forward(self, src, trg):
        pass

INPUT_DIM = len(SRC_vocab)
OUTPUT_DIM = len(TRG_vocab)
EMB_DIM = 32
HID_DIM = 64

enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec)


In [34]:
def get_ngrams(tokens, n):
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return Counter(ngrams)

def clipped_precision(hyp, ref, n):
    hyp_ngrams = get_ngrams(hyp, n)
    ref_ngrams = get_ngrams(ref, n)
    clipped = sum(min(hyp_ngrams[g], ref_ngrams.get(g,0)) for g in hyp_ngrams)
    total = sum(hyp_ngrams.values())
    return clipped / total if total > 0 else 0

def brevity_penalty(hyp, ref):
    c, r = len(hyp), len(ref)
    return 1 if c > r else math.exp(1 - r/c)

def bleu_score(hyp, ref):
    log_prec = []
    for i in range(1,5):
        p = clipped_precision(hyp, ref, i)
        if p == 0: return 0
        log_prec.append(0.25 * math.log(p))
    geo_mean = math.exp(sum(log_prec))
    bp = brevity_penalty(hyp, ref)
    return bp * geo_mean


In [35]:
def greedy_decode(model, src, max_len=10, sos_idx=1, eos_idx=2):
    with torch.no_grad():
        hidden, cell = model.encoder(torch.tensor([SRC_vocab[w] for w in src]))
    input_token = torch.tensor([sos_idx])
    result = [sos_idx]
    for _ in range(max_len):
        output, hidden, cell = model.decoder(input_token, hidden, cell)
        next_token = output.argmax(1).item()
        result.append(next_token)
        if next_token == eos_idx:
            break
        input_token = torch.tensor([next_token])
    return result


In [36]:
def beam_search(model, src, beam_width=3, max_len=10, sos_idx=1, eos_idx=2):
    with torch.no_grad():
        hidden, cell = model.encoder(torch.tensor([SRC_vocab[w] for w in src]))
    beam = [([sos_idx], 0.0, hidden, cell)]
    completed = []
    for _ in range(max_len):
        new_beam = []
        for seq, score, hidden, cell in beam:
            if seq[-1] == eos_idx:
                completed.append((seq, score))
                continue
            input_token = torch.tensor([seq[-1]])
            output, h, c = model.decoder(input_token, hidden, cell)
            log_probs = F.log_softmax(output, dim=1)
            topk = torch.topk(log_probs, beam_width)
            for i in range(beam_width):
                next_token = topk.indices[0][i].item()
                next_score = score + topk.values[0][i].item()
                new_seq = seq + [next_token]
                new_beam.append((new_seq, next_score, h, c))
        beam = sorted(new_beam, key=lambda x: x[1], reverse=True)[:beam_width]
    if completed:
        best_seq = sorted(completed, key=lambda x: x[1], reverse=True)[0][0]
    else:
        best_seq = beam[0][0]
    return best_seq

In [37]:
for i in range(len(dataset)):
    src, ref = dataset[i]

    greedy = greedy_decode(model, src)
    beam = beam_search(model, src, beam_width=5)

    print("SOURCE:   ", src)
    print("GREEDY:   ", tokens_to_words(greedy, TRG_itos))
    print("BEAM:     ", tokens_to_words(beam, TRG_itos))
    print("REFERENCE:", ref)
    print("-"*50)

SOURCE:    ['hello', 'world', '.']
GREEDY:    ['<sos>', '!', '!', 'hello', '!', 'hello', '!', 'world', '!', 'is', '?']
BEAM:      ['<sos>', 'how', '?', 'a', '.', '.', '.', '.', '.', '.', '.']
REFERENCE: ['hello', 'world', '!']
--------------------------------------------------
SOURCE:    ['how', 'are', 'you', '?']
GREEDY:    ['<sos>', '!', '!', '?', '!', '?', '!', '!', '?', '!', '?']
BEAM:      ['<sos>', '?', 'a', '.', '.', '.', '.', '.', '.', '.', '.']
REFERENCE: ['how', 'do', 'you', 'do', '?']
--------------------------------------------------
SOURCE:    ['i', 'am', 'fine', '.']
GREEDY:    ['<sos>', 'doing', '!', 'is', '?', '.', '.', '.', '.', '.', '.']
BEAM:      ['<sos>', 'a', '.', '.', '.', '.', '.', '.', '.', '.', '.']
REFERENCE: ['i', 'am', 'doing', 'well', '.']
--------------------------------------------------
SOURCE:    ['this', 'is', 'a', 'test']
GREEDY:    ['<sos>', '?', 'a', '.', '.', '.', '.', '.', '.', '.', '.']
BEAM:      ['<sos>', '?', 'a', '.', '.', '.', '.', '.', '.'